In [20]:
# 2_test_benchmark.ipynb
import os
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

BASE_DIR = r"C:\Users\User-NB\OneDrive\Desktop\ML"
DATA_PATH = os.path.join(BASE_DIR, "processed_data", "data_processed.pkl")
RESULT_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULT_DIR, exist_ok=True)

# 1. Load Data
data = joblib.load(DATA_PATH)
X, y, fyear = data["X_bench"], data["y"], data["fyear"]
print(f"Benchmark Features: {len(data['feat_names'])}")

# 2. Split
train_mask = (fyear >= 2019) & (fyear <= 2023)
test_mask = (fyear == 2024)
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# 3. Train Benchmark (Standard LR, No SMOTE)
# C=1e9 代表無懲罰 (Standard)
print("Training Benchmark (Standard LR + No SMOTE)...")
bench_model = LogisticRegression(penalty='l2', C=1e9, class_weight=None, solver='lbfgs', max_iter=2000, random_state=42)
bench_model.fit(X_train, y_train)

# 4. Evaluate
y_prob = bench_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print(f"Benchmark AUC: {auc:.4f}")
print(f"Benchmark Brier: {brier:.4f}")

# 5. Save
joblib.dump({"y_test": y_test, "y_prob": y_prob, "auc": auc, "model": bench_model},
            os.path.join(RESULT_DIR, "bench_results.pkl"))

Benchmark Features: 22
Training Benchmark (Standard LR + No SMOTE)...
Benchmark AUC: 0.7188
Benchmark Brier: 0.0360


['C:\\Users\\User-NB\\OneDrive\\Desktop\\ML\\results\\bench_results.pkl']